In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('default')

from agent.components import RASK
from agent.components.commons import ServiceType
from agent.components.commons import SloSet
from notebooks.summersoc.globals import PATH_METRICS_DEMO_EXPLORE


services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SloSet.DEFAULT, SloSet.HIGH_PERF, SloSet.LOW_COST, SloSet.HIGH_QUALITY]

## **Monitor**: Collecting Service Telemetry

This figure shows the networking across the MUDAP platform, mainly relevant for the _monitoring_ and _execution_ phase. An agent can then query data from the time-series DB, infer an action, and orchestrate them over the service API.

![Monitoring Infrastructure](img/MUDAP_Architecture.jpg)

### References

[docker-compose.yml](https://github.com/borissedlak/elastic-workbench/blob/main/docker-compose.yml) describes the monitoring infrastructure (= Prometheus, Grafana, and cAdvisor), and the processing services.

[prometheus.yml](https://github.com/borissedlak/elastic-workbench/blob/main/config/prometheus.yml) defines the endpoints for scraping metrics and the scraping interval.

[IoTServices.py](https://github.com/borissedlak/elastic-workbench/blob/main/iot_services/IoTService.py) creates a Prometheus server within each service and exposes the processing metrics.

[PrometheusClient.py](https://github.com/borissedlak/elastic-workbench/blob/main/PrometheusClient.py) allows the scaling agent to query the states of the processing services

In [2]:
df_explore = pd.read_csv(PATH_METRICS_DEMO_EXPLORE)
df_explore_preprocessed = RASK.preprocess_data(df_explore)
df_explore_preprocessed = df_explore_preprocessed[['timestamp', 'service_type', 'cores', 'data_quality', 'model_size', 'max_tp']]

Using the PrometheusClient.py we can query the metrics from the time-series DB (Prometheus). In our case, the scaling agent pulls the metrics every 10s, this means the current parameter assignments, and the rolling average of the system performance.

Note, that the metrics cover the concurrent performance across all three services and that the CV service has one additional parameter (_model_size_).

In [3]:
df_explore_preprocessed

,timestamp,service_type,cores,data_quality,model_size,max_tp
0,2025-10-11 13:06:19.565816,elastic-workbench-qr-detector,2.67,550,-1.0,120.000000
1,2025-10-11 13:06:19.568908,elastic-workbench-cv-analyzer,2.67,224,3.0,1.533742
2,2025-10-11 13:06:19.571773,elastic-workbench-pc-visualizer,2.67,30,1.0,200.000000
3,2025-10-11 13:06:29.558492,elastic-workbench-qr-detector,1.00,722,-1.0,23.809524
4,2025-10-11 13:06:29.562338,elastic-workbench-cv-analyzer,6.00,297,2.0,6.410256
...,...,...,...,...,...,...
85,2025-10-11 13:10:59.580181,elastic-workbench-cv-analyzer,2.00,227,1.0,4.115226
86,2025-10-11 13:10:59.588945,elastic-workbench-pc-visualizer,1.00,23,1.0,250.000000
87,2025-10-11 13:11:09.576749,elastic-workbench-qr-detector,3.09,553,-1.0,142.857143
88,2025-10-11 13:11:09.579528,elastic-workbench-cv-analyzer,4.08,228,2.0,4.255319
